In [1]:
from pathlib import Path
import os
import subprocess
import sys

repo_name = "flipkart-wired-x-campus-node"
cwd = Path.cwd()
if (cwd / ".git").exists() and cwd.name == repo_name:
    repo = cwd
else:
    base = Path("/content") if Path("/content").exists() else cwd
    repo = base / repo_name
    if (repo / ".git").exists():
        subprocess.run(["git", "-C", str(repo), "pull", "--ff-only", "-q"], check=True)
    else:
        subprocess.run(
            ["git", "clone", "-q", "https://github.com/mba25015-maker/flipkart-wired-x-campus-node.git", str(repo)],
            check=True,
        )

os.chdir(repo)
if os.environ.get("WIRED_SKIP_INSTALL") != "1":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print(f"Repository ready: {repo}")

Repository ready: /Users/dubsey/Documents/Codex/2026-08-29/referenced-chatgpt-conversation-this-is-an/outputs/flipkart-wired-x-campus-node


## What this notebook proves

This notebook reproduces the fulfilment arithmetic behind the adopted delivery cost and topology.

It separates the volume-weighted city gig leg from the fixed-roster in-gate leg, shows the batching and SLA logic, and compares the per-gate base case with pooling as a conditional upside.

Expected headline result: **Rs17.61/order = Rs14.25 city leg + Rs3.36 in-gate leg, using 2 runners × 3 gates.**

In [2]:
import subprocess, sys
subprocess.run([sys.executable, "Model/sla.py"], check=True)
subprocess.run([sys.executable, "Model/fleet_mix.py"], check=True)

                            THE PROMISE WE CAN ACTUALLY KEEP                            
Cluster 1,400 orders/day across 3 campuses -> 25.9 orders/hour at one gate

state           orders/hr  batch (dyn)  batch wait   AVG SLA    LAST  runners  Rs/order
----------------------------------------------------------------------------------------
Trough                6.5            1         9.3      25.7    31.3      1.2      64.8
Average              25.9            2         4.6      21.6    27.8      2.7      33.0
Peak (4x)           103.7           10         5.8      27.1    37.7      3.7       7.7
Exam night (6x)      155.6           12         4.6      27.1    38.8      5.1       6.6
----------------------------------------------------------------------------------------
At trough the city leg is uneconomic (Rs51.2/order at batch 1: one trip, one order).
The in-gate roster is sized to DAILY volume by fleet_mix.optimal_roster, which declines
to add a runner whose residual cannot clear

                     WHY THE LABOUR CLASS DOMINATES THE VEHICLE                     
Gig rider, per ACTIVE hour       Rs   168   bears own vehicle + fuel (25% of gross)
Employed runner, per ROSTERED hr Rs    72   bicycle or on foot, no fuel
                                 --------
Ratio                               2.3x

The intra-campus leg is the ONLY leg that can be served by the cheaper class,
because it is the only leg that never leaves the boundary.

              INTRA-CAMPUS LEG: COST PER ORDER BY MODE AND BATCH SIZE               
Mode                        access  speed      n=1      n=2      n=3      n=4      n=8     n=12
------------------------------------------------------------------------------------
Petrol 2W (incumbent)           NO    20k     28.3     17.6     14.0     12.2       --       --
E-2W                           yes    20k     28.3     17.6     14.0     12.2       --       --
Cycle                          yes    12k     18.6     11.0      8.4       --  

CompletedProcess(args=['/Library/Developer/CommandLineTools/usr/bin/python3', 'Model/fleet_mix.py'], returncode=0)

## How to read the result

Use the SLA output for the service-time identity and the fleet output for roster, gig, pooling, and utilisation economics.

The city leg scales with volume; the in-gate leg is roster cost divided by daily volume. Pooling is not the base case unless the pilot establishes cross-gate movement and reprices repositioning.